<!-- @format -->

# Adult Income Prediction


In [1]:
# Import required libraries and setup module path
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import modules.eda as eda
import modules.preprocessing as preprocessing

url = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"
df = eda.load_data(url)


Dataset shape: (48842, 15)
   age  workclass  fnlwgt     education  educational-num      marital-status  \
0   25    Private  226802          11th                7       Never-married   
1   38    Private   89814       HS-grad                9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm               12  Married-civ-spouse   
3   44    Private  160323  Some-college               10  Married-civ-spouse   
4   18          ?  103497  Some-college               10       Never-married   

          occupation relationship   race  gender  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                  ?    Own-child  White  Female             0             0   

   hours-pe

<!-- @format -->

## 2. Data Pre-processing

Sau bước EDA, nhóm tiến hành tiền xử lý dữ liệu để chuẩn bị cho giai đoạn huấn luyện mô hình.  
Các bước chính gồm:

- chuẩn hóa missing value,
- loại bỏ đặc trưng dư thừa,
- chuẩn bị tập đặc trưng và biến mục tiêu,
- chia train/test,
- xây dựng và so sánh các cấu hình preprocessing cho numerical và categorical features.


<!-- @format -->

### 2.1. Missing value standardization

Kết quả cho thấy dữ liệu thiếu không xuất hiện dưới dạng `NaN` ngay từ đầu mà được mã hóa bằng ký hiệu `?`.  
Sau khi thay thế, các cột có missing value là:

- `workclass`
- `occupation`
- `native-country`

Tỷ lệ thiếu ở các cột này không quá lớn, vì vậy nhóm ưu tiên giữ lại dữ liệu và sẽ xử lý bằng imputation trong pipeline thay vì loại bỏ toàn bộ các dòng.


In [2]:
# [Preprocessing 2.1] Replace '?' with NaN
df_prep = preprocessing.standardize_missing(df)


Missing values after standardization:
age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64


<!-- @format -->

### 2.2. Remove redundant and non-predictive features

Dựa trên kết quả EDA, nhóm loại bỏ hai cột sau:

- `education`: do trùng lặp thông tin với `educational-num`
- `fnlwgt`: đây là sampling weight của điều tra dân số, không phải đặc trưng mô tả trực tiếp cá nhân và thường không mang nhiều ý nghĩa dự đoán trong bài toán phân loại thu nhập

Việc loại bỏ hai cột này giúp giảm dư thừa thông tin và làm tập đặc trưng phù hợp hơn cho mô hình.


In [3]:
# [Preprocessing 2.2] Drop redundant / non-predictive features
df_prep = preprocessing.drop_redundant_features(df_prep, cols_to_drop=['education', 'fnlwgt'])


Shape after dropping: (48842, 13)
Remaining columns: ['age', 'workclass', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


<!-- @format -->

### 2.3. Split features and target

Sau khi tách dữ liệu:

- tập đặc trưng `X` gồm 12 biến,
- biến mục tiêu `y` là `income`.

Phân phối của biến mục tiêu cho thấy số mẫu thuộc lớp `<=50K` lớn hơn đáng kể so với lớp `>50K`, nghĩa là bài toán có hiện tượng mất cân bằng lớp nhẹ.  
Vì vậy, ở bước chia train/test, nhóm sẽ dùng `stratify=y` để giữ phân phối lớp tương đối ổn định giữa hai tập.


In [4]:
# [Preprocessing 2.3] Split features and target
X, y = preprocessing.split_features_target(df_prep, target_col='income')


X shape: (48842, 12)
y shape: (48842,)

Target distribution:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


<!-- @format -->

### 2.4. Identify numerical and categorical features

Các biến được chia thành hai nhóm để phục vụ bước tiền xử lý:

- **Numerical features**: `age`, `educational-num`, `capital-gain`, `capital-loss`, `hours-per-week`
- **Categorical features**: `workclass`, `marital-status`, `occupation`, `relationship`, `race`, `gender`, `native-country`

Việc tách riêng hai nhóm này giúp xây dựng pipeline phù hợp:

- biến số sẽ được xử lý theo các cấu hình preprocessing khác nhau,
- biến phân loại sẽ được xử lý missing value và encoding trước khi đưa vào mô hình.


In [5]:
# [Preprocessing 2.4] Identify numerical and categorical features
num_features, cat_features = preprocessing.identify_feature_types(X)


Numerical features : ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']


<!-- @format -->

### 2.5. Train-test split

Dữ liệu được chia theo tỷ lệ **80/20** cho train và test.  
Việc sử dụng `stratify=y` giúp giữ phân phối lớp của biến mục tiêu gần như giống nhau giữa hai tập.

Điều này đặc biệt quan trọng với bài toán phân loại có mất cân bằng lớp, vì giúp việc đánh giá mô hình ở tập test đáng tin cậy hơn.


In [6]:
# [Preprocessing 2.5] Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

X_train shape: (39073, 12)
X_test shape : (9769, 12)

Train target distribution:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64

Test target distribution:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64


<!-- @format -->

### 2.6. Define preprocessing configurations

Để việc tiền xử lý có ý nghĩa hơn, nhóm xây dựng nhiều cấu hình preprocessing khác nhau thay vì chỉ dùng một pipeline cố định.

Các cấu hình được thay đổi theo ba yếu tố chính:

- cách xử lý missing value ở biến categorical,
- cách scaling biến numerical,
- và giữ nguyên One-Hot Encoding cho các biến categorical không có thứ tự.

Lưu ý: các biến numerical trong bộ dữ liệu này không có missing value, nên không cần áp dụng numerical imputation ở bước so sánh này.


In [7]:
# [Preprocessing 2.6] Define multiple preprocessing configurations
preprocessing_configs = preprocessing.get_preprocessing_configs(num_features, cat_features)


Available preprocessing configurations:
 - config_1_onehot_mostfreq_standard
 - config_2_onehot_constant_standard
 - config_3_onehot_mostfreq_minmax
 - config_4_onehot_mostfreq_noscale


<!-- @format -->

### 2.7. Compare preprocessing configurations

Kết quả cho thấy các cấu hình preprocessing tạo ra số chiều đầu ra khác nhau:

- các cấu hình dùng **One-Hot Encoding** làm tăng số lượng đặc trưng,
- cấu hình dùng giá trị hằng `"Missing"` cho categorical missing có thể làm tăng thêm số category sau encoding,
- các lựa chọn scaling không làm thay đổi số chiều nhưng có thể ảnh hưởng đến hiệu năng mô hình ở bước sau.

Điều này cho thấy lựa chọn preprocessing có ảnh hưởng trực tiếp đến cách biểu diễn dữ liệu đầu vào cho mô hình.


<!-- @format -->

### 2.9. Evaluate preprocessing configurations with Logistic Regression

Để chọn cấu hình preprocessing phù hợp, nhóm sử dụng Logistic Regression như một mô hình đánh giá cố định và so sánh hiệu năng của các cấu hình theo các chỉ số classification.


In [8]:
# [Preprocessing 2.9] Compare preprocessing configurations using Logistic Regression
results_df = preprocessing.compare_preprocessing_configs(
    preprocessing_configs, X_train, X_test, y_train, y_test, pos_label='>50K'
)


                       configuration  n_features  accuracy  precision  recall  \
0  config_2_onehot_constant_standard          91    0.8554     0.7461  0.5997   
1   config_4_onehot_mostfreq_noscale          88    0.8525     0.7387  0.5937   
2  config_1_onehot_mostfreq_standard          88    0.8522     0.7375  0.5937   
3    config_3_onehot_mostfreq_minmax          88    0.8526     0.7440  0.5855   

   f1_score  
0    0.6649  
1    0.6583  
2    0.6578  
3    0.6553  


<!-- @format -->

### 2.10. Select best preprocessing configuration

Dựa trên kết quả so sánh với Logistic Regression, nhóm chọn cấu hình preprocessing tốt nhất để sử dụng cho các bước tiếp theo.


In [9]:
# [Preprocessing 2.10] Select best preprocessing configuration
best_name, best_preprocessor, X_train_best, X_test_best = preprocessing.select_best_config(
    preprocessing_configs, results_df, X_train, X_test
)


Best config selected : config_2_onehot_constant_standard
X_train_best shape   : (39073, 91)
X_test_best shape    : (9769, 91)


<!-- @format -->

Dựa trên kết quả so sánh với Logistic Regression, nhóm chọn `config_2_onehot_constant_standard` làm cấu hình preprocessing tốt nhất cho các bước tiếp theo vì đạt F1-score cao nhất trong các cấu hình đã thử.


<!-- @format -->

### Preprocessing Summary

Sau bước tiền xử lý, nhóm đã:

- Thay thế `?` bằng `NaN` để chuẩn hóa missing value
- Loại bỏ `education` (trùng với `educational-num`) và `fnlwgt` (sampling weight)
- Chia train/test theo tỷ lệ 80/20 với `stratify=y`
- Thử nghiệm 4 cấu hình preprocessing và chọn `config_2_onehot_constant_standard` dựa trên F1-score

Dữ liệu sau preprocessing có **91 đặc trưng**, sẵn sàng cho bước huấn luyện mô hình.
